# Auswertung eines Evaluationslaufs

Dieses Notebook ist ein **offline Analysewerkzeug**: Es liest ausschließlich
gespeicherte Artefakte aus `evaluation/runs/<run_id>/` (Manifest, Plan,
Generierungen, Checks, Bewertungen). „Run All“ löst **keine** API-Aufrufe
aus und benötigt keine Zugangsdaten.

Hinweise:
- Laufdaten nicht committen (`runs/` ist git-ignoriert).
- Ohne importierte Bewertungen (`ratings.jsonl`) bleibt der fachliche
  Teil ausdrücklich unvollständig.
- Markierte Demonstrationsläufe (`allow_unverified_cases=true`) sind
  Werkzeugprüfungen, nicht Forschungsergebnisse.

In [ ]:
import json
import statistics
from collections import defaultdict
from pathlib import Path

RUN_DIR = Path("evaluation/runs/pilot-001")  # <— Lauf auswählen

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
plan = [r for r in read_jsonl(RUN_DIR / "plan.jsonl") if r.get("record_type") == "job"]
exclusions = [r for r in read_jsonl(RUN_DIR / "plan.jsonl") if r.get("record_type") == "exclusion"]
generations = [r for r in read_jsonl(RUN_DIR / "generations.jsonl") if r.get("execution_source") == "live_tutor_api"]
checks = read_jsonl(RUN_DIR / "checks.jsonl")
ratings = read_jsonl(RUN_DIR / "reviews" / "ratings.jsonl")
print("Run:", manifest["run_id"], "· Experiment:", manifest["experiment_id"])

In [ ]:
print("Geplante Jobs:", len(plan), " · Ausschlüsse:", len(exclusions))
print("Hashes:")
for key, value in manifest["hashes"].items():
    print(f"  {key}: {value[:16]}…")
print("Git:", manifest.get("git"))
print("Telemetrie-Limits:")
for key, value in manifest.get("telemetry_limits", {}).items():
    print(f"  {key}: {value}")
if manifest.get("allow_unverified_cases"):
    print("\\n!!! DEMONSTRATIONSLAUF: Korpus enthält nicht verifizierte Fälle — keine fachlichen Schlüsse.")

In [ ]:
outcomes = defaultdict(int)
durations = []
for record in generations:
    outcomes[record.get("outcome", "unbekannt")] += 1
    if record.get("duration_ms") is not None:
        durations.append(record["duration_ms"])

print("Durchlauf:")
for outcome, count in sorted(outcomes.items()):
    print(f"  {outcome}: {count}")
if durations:
    durations.sort()
    p95 = durations[min(len(durations) - 1, int(0.95 * len(durations)))]
    print("Dauer (ms): median", int(statistics.median(durations)), "· p95", p95)

fehler_rows = [r for r in generations if r.get("outcome") != "success"]
print("\\nFehlgeschlagene/unbekannte Versuche (werden nicht still entfernt):", len(fehler_rows))
for record in fehler_rows[:10]:
    print(" ", record.get("job_id"), "→", record.get("outcome"), (record.get("safe_error") or "")[:80])

In [ ]:
checks_by_attempt = defaultdict(list)
for check in checks:
    checks_by_attempt[check["attempt_id"]].append(check)

rows = []
grouped = defaultdict(list)
for record in generations:
    if record.get("outcome") == "success":
        grouped[(record.get("profile_id"), record.get("hint_level"))].append(record)

for (profile_id, level), records in sorted(grouped.items()):
    fails = defaultdict(int)
    present = prohibited = 0
    for record in records:
        for check in checks_by_attempt.get(record["attempt_id"], []):
            if check.get("status") == "fail":
                fails[check["check_id"]] += 1
            if check["check_id"] == "final_answer_disclosure":
                if check.get("evidence", {}).get("present"):
                    present += 1
                if check.get("status") == "fail":
                    prohibited += 1
    rows.append((profile_id, level, len(records), dict(fails), present, prohibited))

print(f"{'profil':22} {'stufe':>5} {'n':>4} {'hintprobleme':>13} {'loesung':>8} {'unzulaessig':>12}")
for profile_id, level, n, fails, present, prohibited in rows:
    print(f"{profile_id:22} {level:>5} {n:>4} {sum(fails.values()):>13} {present:>8} {prohibited:>12}")
print("\\nAchtung: 'loesung' zählt Treffer; auf Stufe 4 ist das erlaubt.",
      "Fehlende Treffer beweisen keine Abwesenheit von Lösungsverrat.")

In [ ]:
ratings_by_attempt = defaultdict(list)
for rating in ratings:
    ratings_by_attempt[rating["attempt_id"]].append(rating)

def median_helpfulness(record):
    values = [r["ratings"].get("hilfreichkeit_naechster_schritt")
              for r in ratings_by_attempt.get(record["attempt_id"], [])
              if r["ratings"].get("hilfreichkeit_naechster_schritt") is not None]
    return statistics.median(values) if values else None

by_key = defaultdict(dict)
for record in generations:
    if record.get("outcome") == "success":
        by_key[(record.get("case_id"), record.get("hint_level"), record.get("repetition"))][record.get("profile_id")] = record

deltas = defaultdict(list)
for (case_id, level, repetition), profiles in by_key.items():
    base = median_helpfulness(profiles.get("base"))
    for profile_id, record in profiles.items():
        if profile_id == "base" or base is None:
            continue
        value = median_helpfulness(record)
        if value is not None:
            deltas[profile_id].append(value - base)

print("Paarvergleiche gegen 'base' (Hilfreichkeit, Median der Deltas):")
for profile_id, values in sorted(deltas.items()):
    print(f"  {profile_id}: Δ {statistics.median(values):+.1f} (n={len(values)})")
if not any(deltas.values()):
    print("  (Noch keine Bewertungen importiert — review-export/review-import nutzen.)")

In [ ]:
print("Beispiele je Profil (erste Antwort, zur Inspektion — nicht als Bewertung):")
seen = set()
for record in generations:
    profile_id = record.get("profile_id")
    if record.get("outcome") != "success" or profile_id in seen:
        continue
    seen.add(profile_id)
    hint = (record.get("returned") or {}).get("hint", "")
    print("\\n===", profile_id, "· Stufe", record.get("hint_level"), "· Fall", record.get("case_id"))
    print(hint[:600])

## Grenzen

- Standardfilter: nur `execution_source == "live_tutor_api"`; Mock-/Fixture-Daten zählen nicht.
- Fachliche Schlüsse erfordern importierte Bewertungen + verifizierte Fälle.
- Tokenverbrauch, interne Upstream-Versuche und `finish_reason` laut Manifest unbekannt.
- Kleinere Korpora: deskriptiv auswerten, Paarvergleiche innerhalb derselben Fälle,
  Wiederholungen nicht als unabhängige Aufgaben behandeln.